# Digital Crisis and Misinformation Detection System Pipeline

**Project:** Final Project – MBMB Melaka  
**Pipeline:** Data Loading → Bilingual NLP Cleaning → Multi-Model Tournament (XGBoost, SVM, LR) → Power BI Export

This notebook processes **64,000+ bilingual articles** (Malay & English) collected from:
- Academic Malay NLP Dataset (Bernama, Astro Awani, Sinar Harian, MalCov)
- Global English Fake News Dataset (ISOT / Hugging Face)
- Live-scraped articles from Sebenarnya.my


In [ ]:
# Run the web scraper to fetch fresh news from Sebenarnya.my
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os

def scrape_sebenarnya(max_pages_per_category=2):
    categories = [
        'nasional/bencana',
        'nasional/keselamatan',
        'sosial/jenayah',
        'sosial/kesihatan'
    ]
    
    base_url = 'https://sebenarnya.my/category/'
    headers = {'User-Agent': 'Mozilla/5.0'}
    
    data = []
    
    for category in categories:
        print(f"Scraping category: {category}")
        for page in range(1, max_pages_per_category + 1):
            url = f"{base_url}{category}/page/{page}/" if page > 1 else f"{base_url}{category}/"
            print(f"  Fetching {url}")
            
            try:
                response = requests.get(url, headers=headers, timeout=10)
                if response.status_code != 200:
                    break
                
                soup = BeautifulSoup(response.text, 'html.parser')
                articles = soup.find_all('h3', class_='entry-title')
                
                for a in articles:
                    link_tag = a.find('a')
                    if not link_tag:
                        continue
                    
                    article_url = link_tag.get('href')
                    title = link_tag.get('title') or link_tag.get_text(strip=True)
                    
                    try:
                        art_res = requests.get(article_url, headers=headers, timeout=10)
                        art_soup = BeautifulSoup(art_res.text, 'html.parser')
                        
                        content_div = art_soup.find('div', class_='td-post-content')
                        text = content_div.get_text(separator=' ', strip=True) if content_div else ''
                        
                        date_tag = art_soup.find('time', class_='entry-date')
                        date = date_tag.get('datetime') if date_tag else ''
                        
                        data.append({
                            'title': title,
                            'url': article_url,
                            'category': category.split('/')[1],
                            'date': date,
                            'text': text
                        })
                        time.sleep(0.5)
                    except Exception as e:
                        print(f"    Failed to fetch article {article_url}: {e}")
                        
            except Exception as e:
                print(f"  Failed to fetch category page {url}: {e}")
    
    df = pd.DataFrame(data)
    csv_path = '../data/raw/data_fake_massive.csv'
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    df.to_csv(csv_path, index=False, encoding='utf-8')
    print(f"Scraping complete. Saved {len(df)} records to {csv_path}")

print("Scraping fresh fake news from Sebenarnya.my...")
scrape_sebenarnya(max_pages_per_category=2)


## 1. Load All Datasets

In [8]:
import pandas as pd
import numpy as np
import os
import warnings
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

print('Loading datasets...')
progress = tqdm(total=3, desc='Loading Sources')

# 1A. Academic Malay Dataset (label: 1=Real, 0=Fake)
df_malay = pd.read_pickle('../data/raw/academic_malay_dataset.pkl')
df_malay = df_malay[['news', 'label']].copy()
df_malay.columns = ['text', 'label']
df_malay['label'] = df_malay['label'].map({1: 'Real', 0: 'Fake'})
df_malay['source'] = 'Academic Malay'
progress.set_postfix_str(f'Academic Malay: {len(df_malay)} articles')
progress.update(1)

# 1B. Global English Dataset (label: 1=Real, 0=Fake)
df_english = pd.read_csv('../data/raw/data_english_global.csv')
df_english = df_english[['text', 'label']].copy()
df_english['label'] = df_english['label'].map({1: 'Real', 0: 'Fake'})
df_english['source'] = 'Global English'
progress.set_postfix_str(f'Global English: {len(df_english)} articles')
progress.update(1)

# 1C. Live Scraped Fake News from Sebenarnya.my
df_scraped = pd.read_csv('../data/raw/data_fake_massive.csv')
df_scraped = df_scraped[['text', 'label']].copy()
df_scraped['source'] = 'Sebenarnya.my'
progress.set_postfix_str(f'Sebenarnya.my: {len(df_scraped)} articles')
progress.update(1)
progress.close()

# Combine all
df_all = pd.concat([df_malay, df_english, df_scraped], ignore_index=True)
df_all = df_all.dropna(subset=['text'])
df_all = df_all[df_all['text'].str.len() > 50]
print(f"\nTOTAL COMBINED DATASET: {len(df_all)} articles")
print(df_all['label'].value_counts())
print(f"Sources: {df_all['source'].value_counts().to_dict()}")


Loading datasets...


Loading Sources:   0%|          | 0/3 [00:00<?, ?it/s]


TOTAL COMBINED DATASET: 63955 articles
label
Real    38210
Fake    25745
Name: count, dtype: int64
Sources: {'Academic Malay': 37463, 'Global English': 24248, 'Sebenarnya.my': 2244}


## 2. Advanced Bilingual NLP Preprocessing
- Remove special characters, URLs, HTML tags
- Lowercase normalization
- Remove both **English AND Malay** stopwords
- Filter extremely short documents


In [9]:
import re
from tqdm.auto import tqdm
tqdm.pandas(desc='Cleaning Articles')

# Comprehensive bilingual stopword list
malay_stopwords = {
    'yang', 'di', 'dan', 'itu', 'dengan', 'untuk', 'tidak', 'ini', 'dari',
    'pada', 'dalam', 'ke', 'akan', 'oleh', 'juga', 'telah', 'ada', 'adalah',
    'kepada', 'sebagai', 'mereka', 'kita', 'kami', 'ia', 'atau', 'bahawa',
    'boleh', 'bagi', 'serta', 'apa', 'daripada', 'lebih', 'banyak', 'lagi',
    'apabila', 'seperti', 'satu', 'dua', 'tiga', 'sudah', 'hanya', 'setelah',
    'masih', 'semua', 'belum', 'antara', 'tanpa', 'bukan', 'begitu', 'kata',
    'orang', 'tahun', 'hari', 'pun', 'nak', 'tak', 'lah', 'kan', 'jer',
    'tu', 'ni', 'dah', 'kena', 'macam', 'bila', 'mana', 'siapa', 'kenapa',
    'bagaimana', 'berapa', 'sebuah', 'seorang', 'tersebut', 'iaitu', 'yakni',
    'tetapi', 'namun', 'walau', 'meskipun', 'walaupun', 'kerana', 'sebab',
    'jika', 'kalau', 'supaya', 'agar', 'hingga', 'sehingga', 'sambil',
    'selain', 'selepas', 'sebelum', 'semasa', 'ketika', 'manakala', 'sedangkan',
    'malah', 'bahkan', 'kini', 'sini', 'sana', 'mahu', 'hendak', 'perlu',
    'harus', 'dapat', 'bisa', 'sangat', 'amat', 'terlalu', 'agak', 'cukup',
    'paling', 'sekali', 'setiap', 'sesuatu', 'segala', 'para', 'beberapa'
}

english_stopwords = {
    'the', 'a', 'an', 'is', 'are', 'was', 'were', 'be', 'been', 'being',
    'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could',
    'should', 'may', 'might', 'shall', 'can', 'need', 'dare', 'ought',
    'used', 'to', 'of', 'in', 'for', 'on', 'with', 'at', 'by', 'from',
    'as', 'into', 'through', 'during', 'before', 'after', 'above', 'below',
    'between', 'out', 'off', 'over', 'under', 'again', 'further', 'then',
    'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'each',
    'every', 'both', 'few', 'more', 'most', 'other', 'some', 'such', 'no',
    'not', 'only', 'own', 'same', 'so', 'than', 'too', 'very', 'just',
    'because', 'but', 'and', 'or', 'if', 'while', 'about', 'against',
    'this', 'that', 'these', 'those', 'it', 'its', 'he', 'she', 'they',
    'them', 'his', 'her', 'their', 'what', 'which', 'who', 'whom', 'i',
    'me', 'my', 'we', 'us', 'our', 'you', 'your', 'up', 'also', 'said'
}

all_stopwords = malay_stopwords.union(english_stopwords)

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower()
    words = text.split()
    words = [w for w in words if w not in all_stopwords and len(w) > 2]
    return ' '.join(words)

df_all['clean_text'] = df_all['text'].progress_apply(clean_text)
df_all['word_count'] = df_all['clean_text'].apply(lambda x: len(x.split()))

# Drop articles that are too short after cleaning
df_clean = df_all[df_all['word_count'] > 5].copy()
df_clean = df_clean.reset_index(drop=True)

os.makedirs('../data/clean', exist_ok=True)
df_clean.to_csv('../data/clean/data_clean.csv', index=False)

print(f"\nAfter cleaning: {len(df_clean)} articles remain")
print(f"Dropped {len(df_all) - len(df_clean)} articles (too short)")
print(df_clean['label'].value_counts())


Cleaning Articles:   0%|          | 0/63955 [00:00<?, ?it/s]


After cleaning: 63897 articles remain
Dropped 58 articles (too short)
label
Real    38205
Fake    25692
Name: count, dtype: int64


## 3. TF-IDF Vectorization with Bi-grams
Using `ngram_range=(1, 2)` so the AI learns **phrases** like "tidak benar" and "fake news", not just individual words.


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

X = df_clean['clean_text']
y = df_clean['label']

# Advanced TF-IDF with unigrams + bigrams
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.95,
    sublinear_tf=True
)

X_vec = vectorizer.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_vec, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Vocabulary size: {len(vectorizer.vocabulary_)} features")
print(f"Training set: {X_train.shape[0]} articles")
print(f"Test set: {X_test.shape[0]} articles")


Vocabulary size: 10000 features
Training set: 51117 articles
Test set: 12780 articles


## 4. Multi-Model Tournament
Training **4 algorithms** simultaneously and selecting the champion:
1. XGBoost (Industry Standard)
2. Support Vector Machine (SVM)
3. Logistic Regression
4. Random Forest


In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
import pickle
import time
from tqdm.auto import tqdm

# Encode labels for XGBoost (it needs numeric labels)
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

models = {
    'XGBoost': XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        eval_metric='logloss',
        use_label_encoder=False
    ),
    'Support Vector Machine': LinearSVC(random_state=42, max_iter=5000),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
}

results = {}
best_model = None
best_acc = 0
best_name = ''

for name, model in tqdm(models.items(), total=len(models), desc='Training Models'):
    start = time.time()
    
    # XGBoost needs encoded labels
    if name == 'XGBoost':
        model.fit(X_train, y_train_enc)
        y_pred = le.inverse_transform(model.predict(X_test))
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    elapsed = time.time() - start
    acc = accuracy_score(y_test, y_pred)
    
    results[name] = {
        'accuracy': acc,
        'time': elapsed,
        'predictions': y_pred
    }
    
    print(f"  {name}: {acc*100:.2f}% ({elapsed:.1f}s)")
    
    if acc > best_acc:
        best_acc = acc
        best_model = model
        best_name = name

print(f"\n{'='*50}")
print(f"CHAMPION MODEL: {best_name}")
print(f"CHAMPION ACCURACY: {best_acc*100:.2f}%")
print(f"{'='*50}")

print(f"\nDetailed Classification Report ({best_name}):")
print(classification_report(y_test, results[best_name]['predictions']))


Training Models:   0%|          | 0/4 [00:00<?, ?it/s]

  XGBoost: 92.76% (210.7s)
  Support Vector Machine: 93.32% (1.7s)
  Logistic Regression: 93.60% (0.7s)
  Random Forest: 92.64% (50.3s)

CHAMPION MODEL: Logistic Regression
CHAMPION ACCURACY: 93.60%

Detailed Classification Report (Logistic Regression):
              precision    recall  f1-score   support

        Fake       0.94      0.90      0.92      5139
        Real       0.93      0.96      0.95      7641

    accuracy                           0.94     12780
   macro avg       0.94      0.93      0.93     12780
weighted avg       0.94      0.94      0.94     12780



## 5. Professional Visualizations
Generate publication-quality charts for your Final Report and Power BI presentation.


In [12]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 5A. Model Comparison Bar Chart
model_names = list(results.keys())
accuracies = [results[m]['accuracy'] * 100 for m in model_names]
colors = ['#2ecc71' if m == best_name else '#3498db' for m in model_names]
axes[0].barh(model_names, accuracies, color=colors, edgecolor='white', height=0.5)
axes[0].set_xlabel('Accuracy (%)')
axes[0].set_title('Model Comparison (Tournament Results)', fontweight='bold')
for i, v in enumerate(accuracies):
    axes[0].text(v + 0.3, i, f'{v:.2f}%', va='center', fontweight='bold')
axes[0].set_xlim(0, 105)

# 5B. Confusion Matrix
from sklearn.metrics import ConfusionMatrixDisplay
cm = confusion_matrix(y_test, results[best_name]['predictions'])
disp = ConfusionMatrixDisplay(cm, display_labels=['Fake', 'Real'])
disp.plot(ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title(f'Confusion Matrix ({best_name})', fontweight='bold')

# 5C. Dataset Distribution
source_counts = df_clean['source'].value_counts()
axes[2].pie(source_counts.values, labels=source_counts.index, autopct='%1.1f%%',
            colors=['#e74c3c', '#3498db', '#2ecc71'], startangle=90)
axes[2].set_title('Dataset Source Distribution', fontweight='bold')

plt.tight_layout()
os.makedirs('../static', exist_ok=True)
plt.savefig('../static/model_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Charts saved to static/model_results.png")


Charts saved to static/model_results.png


## 6. Feature Importance: Top Fake News Keywords
Shows exactly **which words** the AI uses to detect fake news. Perfect for your Power BI Word Cloud!


In [13]:
feature_names = vectorizer.get_feature_names_out()

# Extract feature importance based on model type
if best_name == 'XGBoost':
    importances = best_model.feature_importances_
    word_scores = pd.DataFrame({'Word': feature_names, 'Score': importances})
elif hasattr(best_model, 'coef_'):
    coefs = best_model.coef_[0] if len(best_model.coef_.shape) > 1 else best_model.coef_
    word_scores = pd.DataFrame({'Word': feature_names, 'Score': np.abs(coefs)})
else:
    # Fallback
    word_scores = pd.DataFrame({'Word': feature_names, 'Score': np.asarray(X_vec.sum(axis=0)).ravel()})

word_scores = word_scores.sort_values('Score', ascending=False)

# Plot Top 20 Keywords
fig, ax = plt.subplots(figsize=(10, 8))
top20 = word_scores.head(20)
ax.barh(top20['Word'][::-1], top20['Score'][::-1], color='#e74c3c', edgecolor='white')
ax.set_xlabel('Importance Score')
ax.set_title('Top 20 Most Important Words for Detecting Fake News', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('../static/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Feature importance chart saved!")


Feature importance chart saved!


## 7. Export Model & Datasets


In [14]:
from tqdm.auto import tqdm

progress = tqdm(total=5, desc='Exporting Files')

# 7A. Save the champion model and vectorizer
os.makedirs('../models', exist_ok=True)

with open('../models/model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
progress.set_postfix_str('model.pkl')
progress.update(1)

with open('../models/vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)
progress.set_postfix_str('vectorizer.pkl')
progress.update(1)

with open('../models/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
progress.set_postfix_str('label_encoder.pkl')
progress.update(1)

# 7B. Dashboard Dataset
df_dashboard = df_clean[['source', 'label', 'word_count']].copy()
if best_name == 'XGBoost':
    y_all_pred = le.inverse_transform(best_model.predict(X_vec))
else:
    y_all_pred = best_model.predict(X_vec)
df_dashboard['predicted_label'] = y_all_pred
df_dashboard['is_correct'] = df_dashboard['label'] == df_dashboard['predicted_label']
df_dashboard.to_csv('../data/clean/Misinformation_Analysis_Dashboard.csv', index=False)
progress.set_postfix_str('Misinformation_Analysis_Dashboard.csv')
progress.update(1)

# 7C. Word Frequencies (Top 100)
top100 = word_scores.head(100).copy()
top100.to_csv('../data/clean/Fake_News_Keyword_Importance.csv', index=False)
progress.set_postfix_str('Fake_News_Keyword_Importance.csv')
progress.update(1)
progress.close()

# Summary
print(f"\n{'='*50}")
print(f"ALL EXPORTS COMPLETE")
print(f"{'='*50}")
print(f"Champion Model: {best_name} ({best_acc*100:.2f}%)")
print(f"Model size: {os.path.getsize('../models/model.pkl') / 1024 / 1024:.2f} MB")
print(f"Dashboard: {len(df_dashboard)} rows")
print(f"Keywords: {len(top100)} words")


Exporting Files:   0%|          | 0/5 [00:00<?, ?it/s]


ALL EXPORTS COMPLETE
Champion Model: Logistic Regression (93.60%)
Model size: 0.08 MB
Dashboard: 63897 rows
Keywords: 100 words
